# Laboratório de Análise de Dados
## Instituição: Universidade Federal do Rio Grande do Sul (UFRGS)

Objetivo: Exploração, tratamento e visualização de dados utilizando pandas e altair.        
Nome: Lucas Vieira Bagolin

### Introdução
- Este notebook apresenta o desenvolvimento prático da análise do dataset público da Olist (Brazilian E-Commerce Public Dataset), a maior loja de departamentos dos marketplaces brasileiros. O objetivo desta atividade vai além da simples construção de gráficos: busca-se aplicar um olhar analítico e investigativo sobre os dados operacionais para extrair inteligência de negócios (Business Intelligence) e identificar possíveis gargalos e anomalias na operação.
- Para isso, o trabalho foi estruturado em uma pipeline completa de Ciência de Dados: cruzamento de múltiplas tabelas relacionais, tratamento de dados inconsistentes, transformações temporais/geográficas e, por fim, a criação de painéis visuais interativos. A escolha da biblioteca Altair baseou-se na sua gramática declarativa, permitindo a construção de visualizações avançadas (como painéis marginais, mapas de densidade e regressões) de forma elegante e otimizada.

### Resumo Executivo das Análises
- Ao longo deste laboratório, explorei 10 hipóteses de negócio divididas em pilares estratégicos. Os principais insights descobertos foram:

#### 1. Sazonalidade e Comportamento do Consumidor       
- Tração vs. Faturamento: O volume de pedidos e a receita bruta caminham juntos, mas picos específicos (como a Black Friday em novembro) mostram um aumento expressivo no volume de vendas no e-commerce.
- O "Horário Nobre": Ao contrário do senso comum, o brasileiro não concentra suas compras de e-commerce à noite ou aos finais de semana. O pico absoluto de acessos e conversões ocorre nos dias úteis (Segunda a Quarta), entre 10h e 16h, em pleno horário comercial.

#### 2. Logística e Distribuição Geográfica      
- Densidade e Distância: A aplicação da fórmula de Haversine e o mapa de dispersão provaram que o volume de vendas está massivamente concentrado no eixo Sul-Sudeste (especialmente SP).
- O Custo do Atraso: A análise de proporção de notas provou quantitativamente que o prazo de entrega não é apenas uma estimativa, mas um contrato rigoroso. Falhas operacionais que geram atrasos são o gatilho quase absoluto para avaliações de Nota 1, destruindo a reputação dos lojistas e da plataforma.

#### 3. Inteligência Financeira     
- Parcelamento e Ticket Médio: O cartão de crédito domina o volume de transações. Confirmamos a hipótese de que existe uma correlação linear entre o número de parcelas escolhidas e o ticket médio do produto, com um salto expressivo a partir da 10ª parcela (indicando compras de alto valor agregado viabilizadas por políticas de crédito).
- Dependência (Curva de Pareto): A plataforma apresenta um alto risco de concentração de renda. Provamos que apenas 20% dos vendedores são responsáveis por 70% de todo o faturamento da Olist, evidenciando a dependência de grandes fornecedores em contraste com uma longa cauda de pequenos lojistas.

#### 4. Auditoria de Qualidade e Produto     
- Anomalias de Frete: O cruzamento de peso vs. frete com linha de regressão permitiu identificar outliers na operação, como itens muito leves com fretes exorbitantes e distorções de frete gratuito.
- O Paradoxo dos Móveis: Ao analisar os extremos de satisfação, identificamos categorias campeãs (Alimentos e Livros) e categorias críticas de alto risco logístico (Artigos de Natal, devido à sazonalidade, e Construção/Ferramentas). Curiosamente, a categoria de Móveis de Sala gabaritou as melhores notas, enquanto Móveis de Escritório afundou nas piores avaliações, sinalizando uma disparidade brutal de qualidade entre os fornecedores dessas duas linhas.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import altair as alt

# Desabilitar o limite de linhas do Altair
alt.data_transformers.disable_max_rows() 

# Configurações de exibição do Pandas
pd.set_option('display.max_columns', None)

# Carregando as tabelas principais
df_orders = pd.read_csv(path + "/olist_orders_dataset.csv")
df_items = pd.read_csv(path + "/olist_order_items_dataset.csv")
df_reviews = pd.read_csv(path + "/olist_order_reviews_dataset.csv")
df_products = pd.read_csv(path + "/olist_products_dataset.csv")
df_customers = pd.read_csv(path + "/olist_customers_dataset.csv")
df_sellers = pd.read_csv(path + "/olist_sellers_dataset.csv")

# Tradução de categorias para facilitar a leitura dos gráficos
df_cat_trans = pd.read_csv(path + "/product_category_name_translation.csv")

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Carrega a tabela principal
df_orders = pd.read_csv(path + "/olist_orders_dataset.csv")

# 2. Faz a amostragem estratificada mantendo a proporção de 'order_status'
amostra_orders, _ = train_test_split(
    df_orders,
    train_size=1000,
    stratify=df_orders['order_status'],
    random_state=42
)

# 3. Guarda as chaves principais para filtrar as próximas tabelas
pedidos_ids = amostra_orders['order_id'].unique()
clientes_ids = amostra_orders['customer_id'].unique()

print(f"Amostra raiz criada com {len(amostra_orders)} pedidos.")

In [ ]:
# 1. Carrega as tabelas de nível 1
df_customers = pd.read_csv(path + "/olist_customers_dataset.csv")
df_items = pd.read_csv(path + "/olist_order_items_dataset.csv")
df_payments = pd.read_csv(path + "/olist_order_payments_dataset.csv")
df_reviews = pd.read_csv(path + "/olist_order_reviews_dataset.csv")

# 2. Filtra usando as chaves guardadas
amostra_customers = df_customers[df_customers['customer_id'].isin(clientes_ids)]
amostra_items = df_items[df_items['order_id'].isin(pedidos_ids)]
amostra_payments = df_payments[df_payments['order_id'].isin(pedidos_ids)]
amostra_reviews = df_reviews[df_reviews['order_id'].isin(pedidos_ids)]

# 3. Guarda as novas chaves (produtos e vendedores) para o próximo nível
produtos_ids = amostra_items['product_id'].unique()
vendedores_ids = amostra_items['seller_id'].unique()

print("Tabelas Clientes, Itens, Pagamentos e Avaliações filtradas.")

In [ ]:
# 1. Carrega as tabelas de nível 2
df_products = pd.read_csv(path + "/olist_products_dataset.csv")
df_sellers = pd.read_csv(path + "/olist_sellers_dataset.csv")

# 2. Filtra usando as chaves de itens
amostra_products = df_products[df_products['product_id'].isin(produtos_ids)]
amostra_sellers = df_sellers[df_sellers['seller_id'].isin(vendedores_ids)]

# 3. Carrega a tabela de tradução (não precisa de filtro, é um dicionário pequeno)
df_translation = pd.read_csv(path + "/product_category_name_translation.csv")

print("Tabelas Produtos, Vendedores e Tradução filtradas.")

In [ ]:
# 1. Pega os CEPs dos clientes e vendedores filtrados
ceps_clientes = amostra_customers['customer_zip_code_prefix'].unique()
ceps_vendedores = amostra_sellers['seller_zip_code_prefix'].unique()

# Une todos os CEPs necessários em uma lista única
ceps_validos = list(set(ceps_clientes) | set(ceps_vendedores))

# 2. Carrega a tabela de geolocalização
df_geo = pd.read_csv(path + "/olist_geolocation_dataset.csv")

# 3. Filtra os CEPs e REMOVE duplicatas (mantendo 1 coordenada por CEP)
amostra_geo = df_geo[df_geo['geolocation_zip_code_prefix'].isin(ceps_validos)]
amostra_geo = amostra_geo.drop_duplicates(subset=['geolocation_zip_code_prefix'])

print("Geolocalização filtrada e deduplicada com sucesso.")

In [ ]:
# 1. Começa juntando Pedidos e Clientes
df_final = pd.merge(amostra_orders, amostra_customers, on='customer_id', how='inner')

# 2. Junta os Itens do pedido
df_final = pd.merge(df_final, amostra_items, on='order_id', how='left')

# 3. Junta as informações dos Produtos associados aos itens
df_final = pd.merge(df_final, amostra_products, on='product_id', how='left')

# 4. Traduz a categoria do produto para o português
df_final = pd.merge(df_final, df_translation, on='product_category_name', how='left')

# 5. Junta os Vendedores associados aos itens
df_final = pd.merge(df_final, amostra_sellers, on='seller_id', how='left')

# 6. Junta os Pagamentos do pedido
df_final = pd.merge(df_final, amostra_payments, on='order_id', how='left')

# 7. Junta as Avaliações do pedido
df_final = pd.merge(df_final, amostra_reviews, on='order_id', how='left')

# 8. Adicionando a latitude/longitude do CLIENTE
# Renomea as colunas da amostra_geo para ficar claro que são do cliente
geo_cliente = amostra_geo[['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng']].copy()
geo_cliente.columns = ['customer_zip_code_prefix', 'cliente_lat', 'cliente_lng']

df_final = pd.merge(df_final, geo_cliente, on='customer_zip_code_prefix', how='left')

# Limpando o ambiente e mostrando o resultado
print(f" SUCESSO! O dataset final tem {df_final.shape[0]} linhas e {df_final.shape[1]} colunas.")
display(df_final.head())

In [ ]:
# 1. Removendo as colunas de texto das avaliações
colunas_para_remover = ['review_comment_title', 'review_comment_message']
df_final = df_final.drop(columns=colunas_para_remover, errors='ignore')
print(f"Colunas removidas. O dataset agora tem {df_final.shape[1]} colunas.")

# 2. Convertendo as colunas de data (que estão como texto/string) para formato DateTime
# Isso é essencial para calcular tempo de entrega, atrasos, etc.
colunas_de_data = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date',
    'review_answer_timestamp'
]

for coluna in colunas_de_data:
    # O errors='coerce' transforma erros de formatação em nulos (NaT) em vez de quebrar o código
    if coluna in df_final.columns:
        df_final[coluna] = pd.to_datetime(df_final[coluna], errors='coerce')

print("Datas convertidas com sucesso.")

# 3. Verificando o volume de dados nulos restantes
print("\n--- VALORES NULOS POR COLUNA ---")
nulos = df_final.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))

In [ ]:
# 1. Verificando o tamanho antes da limpeza
tamanho_antes = len(df_final)
print(f"Tamanho antes da limpeza: {tamanho_antes} linhas.")

# 2. Definindo as colunas que NÃO podem ter valores nulos de jeito nenhum
colunas_criticas = [
    'order_id', 
    'customer_id', 
    'product_id', # Sem produto, a linha do item é inútil
    'price',      # Valor da compra é essencial para quase toda análise
    'order_approved_at', # Garante que o pedido foi realmente processado
    'shipping_limit_date', # Data limite de envio é crucial para análises de logística
    'order_delivered_customer_date', # Data de entrega é essencial para análises de logística e satisfação
    'product_category_name', # Categoria do produto é essencial para análises de vendas e preferências
    'review_id', # Sem avaliação, não podemos analisar satisfação do cliente
    'cliente_lat' # Latitude do cliente é essencial para análises geográficas
]

# 3. Removendo as linhas que possuem NaN em qualquer uma das colunas críticas
df_final = df_final.dropna(subset=colunas_criticas)

# 4. Focar apenas em pedidos que deram certo
# Pedidos 'canceled' ou 'unavailable' costumam ter muitas datas de entrega nulas.
df_final = df_final[df_final['order_status'] == 'delivered']

# 5. Calculando o impacto da limpeza
tamanho_depois = len(df_final)
linhas_removidas = tamanho_antes - tamanho_depois

print(f"Limpeza concluída!")
print(f"Linhas removidas: {linhas_removidas}")
print(f"Tamanho atual do dataset: {tamanho_depois} linhas.")

Variáveis Numéricas: Preço, frete, valor do pagamento, peso do produto, etc...

Variáveis Categóricas: Estado do cliente, status do pedido, categoria do produto, etc...

Variável de Tempo: Várias datas (compra, aprovação, entrega).

In [ ]:
# --- ANÁLISE EXPLORATÓRIA: SAZONALIDADE DE PEDIDOS E FATURAMENTO ---

# 1. Garante que a coluna de data esteja no formato DateTime
df_final['order_purchase_timestamp'] = pd.to_datetime(df_final['order_purchase_timestamp'])

# 2. Cria a coluna de Valor Total Gasto (Preço + Frete)
df_final['valor_total'] = df_final['price'] + df_final['freight_value']

# 3. Agrupando os dados por mês (reuso da lógica anterior com acréscimo de soma)
sazonalidade_completa = df_final.groupby(df_final['order_purchase_timestamp'].dt.to_period('M')).agg({
    'order_id': 'nunique',       # Contagem de pedidos únicos
    'valor_total': 'sum'          # Soma do valor total gasto
}).reset_index()

# 4. Ajustes de formatação
sazonalidade_completa.columns = ['Mes', 'Total_Pedidos', 'Gasto_Total']
sazonalidade_completa['Mes'] = sazonalidade_completa['Mes'].dt.to_timestamp()

print("Dados agrupados para sazonalidade:")

# --- GERAÇÃO DA VISUALIZAÇÃO COM ALTAIR ---
# Base interativa para que os gráficos compartilhem seleção e tooltips
base = alt.Chart(sazonalidade_completa).encode(
    x=alt.X('Mes', title='Mês da Compra', axis=alt.Axis(format='%b %Y'))
)

# 1. Gráfico de Linha
line = base.mark_line(color='#1f77b4', size=3).encode(
    y=alt.Y('Total_Pedidos', title='Total de Pedidos Unique'),
    tooltip=[
        alt.Tooltip('Mes', title='Mês', format='%B %Y'),
        'Total_Pedidos'
    ]
).properties(
    title='Volume de Pedidos (Volume)',
    width=600,
    height=250
).interactive()

# 2. Gráfico de Barras
bar = base.mark_bar(color='#d62728', size=20).encode(
    y=alt.Y('Gasto_Total', title='Valor Total Gasto (R$)'),
    tooltip=[
        alt.Tooltip('Mes', title='Mês', format='%B %Y'),
        alt.Tooltip('Gasto_Total', title='Gasto Total (R$)', format='$,.2f')
    ]
).properties(
    title='Gasto Total Mensal (Faturamento)',
    width=600,
    height=150
)

# 3. Concatenando verticalmente
painel_sazonal = (line & bar).configure_title(
    fontSize=14,
    anchor='middle'
)

# Exibindo o resultado limpo de warnings
painel_sazonal

Justificativa de Design: Para responder à pergunta "os meses com mais compradores são obrigatoriamente os meses mais lucrativos?", optei por concatenar verticalmente dois gráficos que compartilham o mesmo eixo temporal (eixo X). Usei uma linha para demonstrar a tendência de adoção (Volume de Pedidos) e barras para quantificar o montante financeiro bruto (Faturamento). Essa escolha evita a sobrecarga cognitiva de múltiplos eixos Y no mesmo espaço e facilita a comparação direta entre as métricas.

Ao cruzar as duas visualizações, extraí as seguintes conclusões de negócio:

A Desconexão entre Volume e Receita (Setembro de 2017): O insight mais valioso desta exploração ocorre no início do segundo semestre de 2017. O gráfico de linhas mostra um volume de pedidos intermediário (abaixo de 50 pedidos únicos), muito distante dos recordes da plataforma. No entanto, o gráfico de barras revela um salto financeiro absoluto e desproporcional neste mesmo mês, ultrapassando a marca dos R$ 18.000. Isso evidencia um pico fortíssimo no ticket médio: a plataforma converteu menos clientes, mas vendeu produtos de altíssimo valor agregado nesta amostra.

Pico de Black Friday: Observa-se um salto expressivo em novembro de 2017. Este é um comportamento clássico do varejo brasileiro provocado pela Black Friday, demonstrando que a plataforma teve forte adesão durante a data promocional.

A Anomalia de Abril de 2017: Um comportamento similar é observado no fechamento do primeiro semestre de 2017. Com apenas cerca de 20 pedidos, o faturamento ultrapassou os R$ 10.000, superando meses seguintes que tiveram um número muito maior de compradores. Esse efeito de ticket médio elevado pode ser um reflexo direto do comportamento de compras de bens mais caros para o Dia das Mães.

Maturidade e Escala (2018): O ano de 2018 indica a estabilização da operação. O volume de vendas abandona a grande volatilidade inicial e estabelece um platô elevado e consistente (oscilando entre 55 e 75 pedidos mensais). O faturamento acompanha essa consistência, operando quase sempre acima de R$ 10.000 mensais, indicando previsibilidade para o negócio.

Limitações de Captação: O término abrupto da captação nas extremidades do eixo temporal (outubro de 2016 e agosto de 2018) não representam falhas do e-commerce, mas sim os limites de início e fim da extração do dataset utilizado.

In [ ]:
# --- ANÁLISE EXPLORATÓRIA: DISTRIBUIÇÃO GEOGRÁFICA DA DEMANDA ---

# 1. Limpeza de anomalias geográficas
# Removemos nulos e filtramos as coordenadas para os limites aproximados do Brasil.
# Isso evita que CEPs digitados incorretamente na base do Olist distorçam o zoom do mapa.
df_mapa = df_final.dropna(subset=['cliente_lat', 'cliente_lng'])
df_mapa = df_mapa[
    (df_mapa['cliente_lat'] < 6) & (df_mapa['cliente_lat'] > -34) &
    (df_mapa['cliente_lng'] < -34) & (df_mapa['cliente_lng'] > -74)
]

# 2. Criando o Mapa de Dispersão (Densidade)
# O uso de 'mercator' projeta os dados num formato de mapa tradicional
mapa = alt.Chart(df_mapa).mark_circle(size=15, opacity=0.3, color='#d62728').encode(
    longitude='cliente_lng:Q',
    latitude='cliente_lat:Q',
    tooltip=[
        alt.Tooltip('customer_city:N', title='Cidade'), 
        alt.Tooltip('customer_state:N', title='Estado')
    ]
).project(
    type='mercator'
).properties(
    title='Densidade Geográfica de Compras',
    width=400,
    height=500
)

# 3. Preparando os dados para o Gráfico de Barras
vendas_estado = df_mapa.groupby('customer_state').size().reset_index(name='Total_Compras')

# 4. Criando o Gráfico de Barras para quantificar os estados
barras = alt.Chart(vendas_estado).mark_bar(color='#1f77b4').encode(
    x=alt.X('Total_Compras:Q', title='Volume de Pedidos'),
    # A ordenação '-x' garante que a barra maior fique no topo
    y=alt.Y('customer_state:N', sort='-x', title='Estado (UF)'),
    tooltip=[
        alt.Tooltip('customer_state:N', title='Estado'),
        alt.Tooltip('Total_Compras:Q', title='Total de Pedidos')
    ]
).properties(
    title='Ranking Absoluto por UF',
    width=250,
    height=500
)

# 5. Concatenando horizontalmente (Operador | no Altair)
painel_geografico = (mapa | barras).configure_title(
    fontSize=14,
    anchor='middle'
)

# Exibindo o resultado interativo
painel_geografico

A visualização combinada acima emprega uma projeção espacial ao lado de uma ordenação quantitativa para diagnosticar a origem da receita do Olist. A escolha de não utilizar polígonos estaduais no mapa, mas sim os pontos geográficos exatos das entregas, permite visualizar a concentração real em pólos urbanos e a dispersão em áreas interioranas.

Hegemonia do Sudeste: O mapa de densidade revela uma mancha avermelhada intensa cobrindo a região Sudeste, com o gráfico de barras confirmando que o estado de São Paulo (SP) detém a esmagadora maioria do volume de compras, seguido por Rio de Janeiro (RJ) e Minas Gerais (MG).

Comportamento das Capitais vs. Interior: O detalhamento dos pontos mostra que, no Nordeste e no Sul, as compras estão fortemente concentradas na faixa litorânea e nas capitais, enquanto o interior do país apresenta grande rarefação.

Implicação Logística: Essa desigualdade de distribuição sugere que estratégias de frete grátis ou otimização de tempo de entrega trarão muito mais retorno financeiro se focadas nas rotas de São Paulo e Rio de Janeiro, onde a adoção do serviço já está estabelecida e provada.

In [ ]:
# --- ANÁLISE: COMO O ATRASO NA ENTREGA AFETA A AVALIAÇÃO DO CLIENTE ---

# 1. Filtrei apenas as linhas que têm as datas e as notas preenchidas
df_analise = df_final.dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date', 'review_score']).copy()

# 2. Convertendo explicitamente para datetime para garantir que a conta funcione
df_analise['order_delivered_customer_date'] = pd.to_datetime(df_analise['order_delivered_customer_date'])
df_analise['order_estimated_delivery_date'] = pd.to_datetime(df_analise['order_estimated_delivery_date'])

# 3. Calculamos o atraso em dias
df_analise['atraso_dias'] = (df_analise['order_delivered_customer_date'] - df_analise['order_estimated_delivery_date']).dt.days

# 4. Criei uma coluna categórica: se o atraso for maior que 0, está atrasado.
df_analise['status_entrega'] = df_analise['atraso_dias'].apply(lambda x: 'Atrasado' if x > 0 else 'No Prazo / Adiantado')

# 5. Agrupei os dados para contar quantas notas de cada tipo existem por status
distribuicao_notas = df_analise.groupby(['status_entrega', 'review_score']).size().reset_index(name='contagem')

# 6. Transformei a contagem bruta em percentual (proporção) dentro de cada status
total_por_status = distribuicao_notas.groupby('status_entrega')['contagem'].transform('sum')
distribuicao_notas['percentual'] = (distribuicao_notas['contagem'] / total_por_status) * 100

print("Dados preparados para o gráfico:")
display(distribuicao_notas.head())


# --- GERAÇÃO DA VISUALIZAÇÃO COM ALTAIR ---

# Criei o gráfico de barras empilhadas
grafico_atraso = alt.Chart(distribuicao_notas).mark_bar().encode(
    # Eixo X com o status da entrega
    x=alt.X('status_entrega:N', title='Status da Entrega', axis=alt.Axis(labelAngle=0)),
    
    # Eixo Y com a porcentagem
    y=alt.Y('percentual:Q', title='Proporção das Notas (%)'),
    
    # Cores indicando a nota (do Vermelho=1 ao Verde=5)
    color=alt.Color('review_score:O', 
                    title='Nota da Avaliação',
                    scale=alt.Scale(
                        domain=[1, 2, 3, 4, 5],
                        range=['#d62728', '#ff7f0e', '#ffbb78', '#98df8a', '#2ca02c']
                    )),
    
    # Tooltip interativo
    tooltip=[
        alt.Tooltip('status_entrega:N', title='Status'),
        alt.Tooltip('review_score:O', title='Nota'),
        alt.Tooltip('percentual:Q', title='Percentual (%)', format='.1f'),
        alt.Tooltip('contagem:Q', title='Volume Absoluto')
    ]
).properties(
    title='O Custo do Atraso: Impacto Logístico na Avaliação do Cliente',
    width=400,
    height=350
).configure_title(
    fontSize=16,
    anchor='middle'
)

# Exibe o gráfico
grafico_atraso

Para visualizar se o atraso na entrega realmente destrói a nota da loja, dividi os pedidos em dois grupos simples: "Atrasado" e "No Prazo / Adiantado". Optei por um gráfico de barras empilhadas em porcentagem. Essa foi uma escolha de design estratégica: como o volume de pedidos no prazo é muito maior do que os atrasados, olhar as porcentagens deixa a comparação justa. Também usei cores que vão do vermelho (nota 1) ao verde (nota 5) para bater o olho e já entender o nível de satisfação.

Olhando para o resultado, podemos chegar às seguintes conclusões:

O peso do atraso (A mancha vermelha): Fica nítido que atrasar a entrega é o caminho mais rápido para receber uma avaliação péssima. Nos pedidos entregues no prazo, a área vermelha (nota 1) é bem pequena. Mas quando olhamos para a barra dos pedidos atrasados, o vermelho toma conta de uma parte enorme do gráfico. O cliente simplesmente não perdoa o atraso.

O básico bem feito (A dominância do verde): Por outro lado, a cor verde (nota 5) é a grande maioria entre os pedidos que chegaram no prazo. Isso mostra que fazer o básico — que é cumprir a data que foi prometida no momento da compra — já garante a nota máxima na maioria das vezes.

Prazo é compromisso: O gráfico prova com dados que a data de entrega calculada pelo sistema não é só um detalhe, é uma promessa. Errar esse prazo custa muito caro para a reputação e para as notas do e-commerce.

In [ ]:
# --- ANÁLISE EXPLORATÓRIA: PREFERÊNCIA DE PAGAMENTO E DISTRIBUIÇÃO DO TICKET ---

# Remover pagamentos não definidos para limpar os dados
df_pagamentos = df_final[df_final['payment_type'] != 'not_defined'].copy()

# 1. Gráfico de Barras (Volume de Transações)
# Mostra a popularidade real de cada método
barras_volume = alt.Chart(df_pagamentos).mark_bar().encode(
    x=alt.X('payment_type:N', title='Método de Pagamento', sort='-y'),
    y=alt.Y('count():Q', title='Volume de Pedidos'),
    color=alt.Color('payment_type:N', legend=None),
    tooltip=[
        alt.Tooltip('payment_type:N', title='Método'),
        alt.Tooltip('count():Q', title='Total de Pedidos')
    ]
).properties(
    title='Preferência de Pagamento (Volume)',
    width=300,
    height=400
)

# 2. Boxplot Ajustado (Distribuição com Zoom)
# Remover o extent='min-max' (volta pro padrão estatístico) 
# e aplicar clip=True com domain para dar zoom sem apagar os outliers do cálculo
boxplot_ajustado = alt.Chart(df_pagamentos).mark_boxplot(size=40, clip=True).encode(
    x=alt.X('payment_type:N', title='Método de Pagamento', sort='-y'),
    y=alt.Y('payment_value:Q', 
            title='Valor do Pagamento (R$)', 
            scale=alt.Scale(domain=[0, 800]) # Foca na faixa onde estão >95% das compras
           ),
    color=alt.Color('payment_type:N', legend=None)
).properties(
    title='Distribuição do Ticket (Zoom até R$ 800)',
    width=300,
    height=400
)

# Concatenando horizontalmente
painel_pagamentos = (barras_volume | boxplot_ajustado).configure_title(
    fontSize=14,
    anchor='middle'
)

display(painel_pagamentos)

Justificativa de Design: Analisar dados financeiros em e-commerce exige cuidado com outliers (compras de altíssimo valor que distorcem as médias). Um gráfico Boxplot isolado sofre de duas deficiências: primeiro, é esmagado pelos valores extremos; segundo, ele omite o volume total de adesão de cada categoria. Para solucionar isso, optei por um painel duplo concatenado horizontalmente. À esquerda, um gráfico de barras quantifica a popularidade (Volume). À direita, o Boxplot revela a distribuição do ticket médio (Valor), ao qual apliquei um clipping visual no eixo Y (limitado a R$ 800) para evidenciar a área interquartil sem remover matematicamente os outliers do cálculo.

Cruzando essas duas visões, concluí que:

A Hegemonia do Cartão de Crédito: O gráfico de barras não deixa dúvidas: o cartão de crédito é a espinha dorsal do e-commerce brasileiro, dominando o volume de transações de forma absoluta.

Elasticidade de Gastos: Observando a "caixa" do cartão de crédito no boxplot ajustado, confirmei que ele não apenas é o mais usado, mas também o método que os clientes utilizam para compras mais caras. Sua área interquartil e a mediana (linha central) são as mais elevadas.

O Papel do Boleto: O boleto se consolida como o segundo método mais popular em volume. Contudo, seu boxplot é ligeiramente mais comprimido em relação ao crédito, indicando uma resistência do consumidor em assumir grandes desembolsos à vista sem a segurança do chargeback do cartão.

Vouchers (Cupons/Vale-Presente): O comportamento do voucher reflete seu uso prático: ele possui uma caixa muito achatada e baixa no boxplot, comprovando que os clientes o utilizam majoritariamente para abater valores fracionados, pequenos itens ou o custo do frete, e raramente para pagar o valor integral de produtos caros.

In [ ]:
# --- ANÁLISE EXPLORATÓRIA: A RELAÇÃO ENTRE O VALOR DA COMPRA E O PARCELAMENTO ---

# Agrupar pelo número de parcelas e tiramos a média do valor pago
df_parcelas = df_final.groupby('payment_installments')['payment_value'].mean().reset_index()

# Gráfico de Barras
barras_parcelas = alt.Chart(df_parcelas).mark_bar(color='#ff7f0e').encode(
    x=alt.X('payment_installments:O', title='Número de Parcelas', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('payment_value:Q', title='Ticket Médio (R$)'),
    tooltip=[
        alt.Tooltip('payment_installments:O', title='Parcelas'),
        alt.Tooltip('payment_value:Q', title='Valor Médio', format='$,.2f')
    ]
).properties(
    title='A Relação entre o Valor da Compra e o Parcelamento',
    width=600,
    height=300
)

display(barras_parcelas)

Quería testar se existe uma relação direta entre o valor do produto e a necessidade de parcelamento. Um gráfico de barras simples, ordenado pelo número de parcelas no eixo X e mostrando a média de gasto no eixo Y, é a forma mais limpa de validar essa hipótese visualmente.

O que os dados nos dizem:   
Correlação Clara: Existe uma escada no gráfico. Fica provado que o ticket médio da compra dita o número de parcelas. Compras pagas em 1 ou 2 vezes têm valores médios muito baixos.

A Barreira das 10 Parcelas: A partir de 10 parcelas, vemos um salto expressivo no ticket médio. Isso indica que a loja possivelmente oferece parcelamento sem juros até 12x, o que atrai consumidores para itens de alto valor agregado, como eletrônicos ou móveis. A facilidade de parcelar é o que viabiliza a venda dos produtos mais caros do catálogo.

In [ ]:
# --- ANÁLISE EXPLORATÓRIA: AS CATEGORIAS DE PRODUTOS MAIS VENDIDAS ---

# 1. Agrupei usando a coluna original em português
df_categorias = df_final.groupby('product_category_name').agg({
    'order_id': 'nunique', # Volume
    'price': 'sum'         # Receita
}).reset_index()

# 2. Limpeza estética do texto (tira underlines e coloca as primeiras letras em maiúsculo)
df_categorias['product_category_name'] = df_categorias['product_category_name'].str.replace('_', ' ').str.title()

# 3. Separa as Top 15 categorias com mais vendas para o gráfico de barras não ficar gigante
top_categorias = df_categorias.nlargest(15, 'order_id')

# --- GERAÇÃO DA VISUALIZAÇÃO COM ALTAIR ---
# Gráfico 1: A Matriz de Dispersão
scatter = alt.Chart(df_categorias).mark_circle(size=80, opacity=0.7, color='#2ca02c').encode(
    x=alt.X('order_id:Q', title='Volume de Vendas'),
    y=alt.Y('price:Q', title='Faturamento Total (R$)'),
    tooltip=[
        alt.Tooltip('product_category_name:N', title='Categoria'),
        alt.Tooltip('order_id:Q', title='Volume Vendido'),
        alt.Tooltip('price:Q', title='Faturamento (R$)', format='$,.2f')
    ]
).properties(
    title='Matriz: Volume vs. Faturamento',
    width=350,
    height=400
).interactive()

# Gráfico 2: Barras Horizontais para leitura rápida
barras = alt.Chart(top_categorias).mark_bar(color='#1f77b4').encode(
    x=alt.X('order_id:Q', title='Volume de Vendas'),
    # sort='-x' garante que a maior barra fique no topo
    y=alt.Y('product_category_name:N', sort='-x', title='Categoria de Produto'), 
    tooltip=[
        alt.Tooltip('product_category_name:N', title='Categoria'),
        alt.Tooltip('order_id:Q', title='Volume Vendido')
    ]
).properties(
    title='Top 15 Categorias (Volume)',
    width=300,
    height=400
)

# Concatenando lado a lado
painel_produtos = (scatter | barras).configure_title(
    fontSize=14,
    anchor='middle'
)

display(painel_produtos)

Justificativa de Design: O gráfico de dispersão (scatter plot) é a ferramenta matemática ideal para cruzar duas variáveis contínuas (Volume vs. Faturamento), criando uma matriz de quadrantes. No entanto, ele oculta os rótulos das categorias, exigindo interação constante. Para solucionar esse gargalo de usabilidade, apliquei uma concatenação horizontal, pareando a matriz com um gráfico de barras focado apenas nas Top 15 categorias. O uso de barras horizontais é a melhor prática de design para facilitar a leitura de rótulos de texto longos, garantindo que o avaliador identifique imediatamente os líderes de venda sem precisar interagir com o painel.

Ao observar o painel pareado, extraí as seguinte conclusões sobre o catálogo:

Os "Motores" do E-commerce: O gráfico de barras evidencia instantaneamente que nichos como Cama Mesa Banho, Beleza Saude e Esporte Lazer dominam o volume de transações. Ao buscar essas categorias na matriz ao lado (os pontos mais distantes no eixo X), confirmamos que elas não apenas giram o estoque, mas também puxam a linha de faturamento da empresa para o alto.

Produtos de "Giro Rápido" (Baixo Ticket): Analisando a matriz de dispersão, notamos categorias que avançam no eixo de Volume (X), mas se mantêm achatadas no eixo de Faturamento (Y). Produtos de utilidades domésticas ou acessórios baratos geralmente caem nesse quadrante: exigem grande esforço logístico, mas geram margens financeiras brutas menores.

Alto Valor Agregado: Pontos isolados que escalam verticalmente no eixo Y, mas sem grande volume no eixo X, revelam os nichos de luxo ou tecnologia (como Relógios Presentes ou Informática). Eles dependem de poucas conversões para impactar substancialmente a receita final do negócio.

In [ ]:
# --- ANÁLISE EXPLORATÓRIA: HORÁRIO NOBRE DE COMPRAS (DIA/HORA) ---

# --- PREPARAÇÃO DOS DADOS --- (Reuso e Limpeza para português)
df_final['hora_compra'] = df_final['order_purchase_timestamp'].dt.hour
df_final['dia_semana'] = df_final['order_purchase_timestamp'].dt.dayofweek # 0=Segunda, 6=Domingo

# Mapeando os dias para português com número para ordenação
dias_map = {0: '1-Seg', 1: '2-Ter', 2: '3-Qua', 3: '4-Qui', 4: '5-Sex', 5: '6-Sáb', 6: '7-Dom'}
df_final['nome_dia'] = df_final['dia_semana'].map(dias_map)

# Agrupando por dia e hora
df_horarios = df_final.groupby(['nome_dia', 'hora_compra'])['order_id'].nunique().reset_index(name='total_pedidos')

# Agrupando por dia (Volume Simples) para o gráfico marginal
df_volume_dia = df_horarios.groupby('nome_dia')['total_pedidos'].sum().reset_index()

# --- GERAÇÃO DO PAINEL COM ALTAIR ---
# Base comum para compartilhamento de eixos
base = alt.Chart(df_horarios).encode(
    x=alt.X('nome_dia:N', title='Dia da Semana', sort='ascending')
)

# 1. O Heatmap Original (Ajustado)
heatmap = base.mark_rect().encode(
    y=alt.Y('hora_compra:O', title='Hora do Dia (0h - 23h)', sort='descending'), # Hora em O (ordinal) para heatmap
    color=alt.Color('total_pedidos:Q', title='Volume', scale=alt.Scale(scheme='blues')),
    tooltip=[
        alt.Tooltip('nome_dia:N', title='Dia'),
        alt.Tooltip('hora_compra:O', title='Hora'),
        alt.Tooltip('total_pedidos:Q', title='Total Pedidos')
    ]
).properties(
    title='Horário Nobre: Detalhado (Dia/Hora)',
    width=350,
    height=350
)

# 2. O Gráfico Marginal de Barras (Volume Simples por Dia)
# Usei a base que compartilha o eixo X (Dia)
barras_marginais = base.mark_bar(color='#1f77b4', size=30).encode(
    y=alt.Y('sum(total_pedidos):Q', title='Volume Total Diário'),
    tooltip=[
        alt.Tooltip('nome_dia:N', title='Dia'),
        alt.Tooltip('sum(total_pedidos):Q', title='Volume Total Diário')
    ]
).properties(
    title='Volume Simples por Dia da Semana',
    width=350,
    height=150
)

# Concatenando verticalmente (operador &) e permitindo interatividade total
# Adicionei o interactive() apenas na concatenação final
painel_horarios = (heatmap & barras_marginais).configure_title(
    fontSize=14,
    anchor='middle'
)

display(painel_horarios)

Justificativa de Design: Optei por um design composto vertical. Concatenei o heatmap original com um gráfico de barras que quantifica o volume simples acumulado por dia da semana, compartilhando o eixo X. Essa decisão analítica permite a observação, em um único painel e com interatividade total, tanto a tendência macro (qual é o melhor dia?) quanto o horário micro (qual é a melhor hora naquele dia?).

Ao analisar este painel avançado, as conclusões ficam muito claras:

Validação da Tendência Útil: O gráfico de barras confirma que a força de vendas do e-commerce Olist está nos dias úteis. Segunda, Terça e Quarta-feira são, de longe, os dias de maior volume de vendas.

O "Pico do Expediente": Cruzando essa informação com o heatmap acima, notamos que as cores escuras (alto volume) se intensificam nesses mesmos dias entre 10h e 16h. Isso quebra o mito de que as pessoas compram mais à noite; na verdade, o pico de consumo acontece durante o horário comercial de trabalho.

Decisão de Marketing: A recomendação estratégica agora tem duas camadas. Se o objetivo é queimar estoque, o marketing deve disparar campanhas nas manhãs de quinta-feira. Se o objetivo é tentar reaquecer um mercado fraco, pode-se tentar promoções relâmpago no sábado à noite, que é o pior momento da semana.

In [ ]:
# --- ANÁLISE EXPLORATÓRIA: AS CATEGORIAS DE PRODUTOS COM AS PIORES E MELHORES AVALIAÇÕES ---

# --- PREPARAÇÃO DOS DADOS --- 
df_final['product_category_name'] = df_final['product_category_name'].str.replace('_', ' ').str.title()

df_notas = df_final.groupby('product_category_name').agg(
    nota_media=('review_score', 'mean'),
    total_avaliacoes=('review_score', 'count')
).reset_index()

# Filtro Crucial de Volume (atenuante de anomalias)
df_notas_validas = df_notas[df_notas['total_avaliacoes'] >= 5]

# Separação dos Extremos
# 1. Top 10 Piores (nsmallest)
piores_categorias = df_notas_validas.nsmallest(10, 'nota_media')
# 2. Top 10 Melhores (nlargest)
melhores_categorias = df_notas_validas.nlargest(10, 'nota_media')

# --- GERAÇÃO DO PAINEL COM ALTAIR ---
# Base comum (sem dados ainda)
base_notas = alt.Chart().mark_bar().encode(
    x=alt.X('nota_media:Q', title='Nota Média (1 a 5)', scale=alt.Scale(domain=[0, 5])),
    tooltip=[
        alt.Tooltip('product_category_name:N', title='Categoria'),
        alt.Tooltip('nota_media:Q', title='Nota Média', format='.1f'),
        alt.Tooltip('total_avaliacoes:Q', title='Total Avaliações')
    ]
).properties(
    width=300, # Gráficos mais estreitos para caber lado a lado
    height=300
)

# Gráfico 1: Piores Notas (Vermelho Alerta)
piores = base_notas.properties(
    data=piores_categorias,
    title='Alerta de Qualidade (Piores)'
).encode(
    y=alt.Y('product_category_name:N', sort='x', title='As 10 Piores Categorias'),
    color=alt.ColorValue('#d62728')
)

# Gráfico 2: Melhores Notas (Verde Sucesso)
melhores = base_notas.properties(
    data=melhores_categorias,
    title='Padrão de Excelência (Melhores)'
).encode(
    y=alt.Y('product_category_name:N', sort='-x', title='As 10 Melhores Categorias'),
    color=alt.ColorValue('#2ca02c')
)

# Concatenando horizontalmente
painel_extremos = (piores | melhores).configure_title(
    fontSize=14,
    anchor='middle'
)

display(painel_extremos)

Justificativa de Design: Para atender ao requisito de comparar o desempenho de todo o catálogo de produtos e superar o limite visual de tentar renderizar mais de 70 categorias simultaneamente — o que resultaria em um gráfico ilegível —, optei por um design de painel de extremos. Concatenei horizontalmente o "Top 10 Melhores Notas" com o "Top 10 Piores Notas", aplicando uma escala de cores divergente clássica (vermelho para alerta, verde para sucesso). Essa decisão cumpre o requisito de design ao criar um contraste imediato, permitindo que o avaliador observe a disparidade de satisfação entre nichos de mercado diferentes.

Cruzando as duas visões, concluí que:       
A Disparidade é Real: Ao exibir os extremos, a comparação é imediata. Enquanto as 10 melhores categorias operam em patamares de excelência (beirando notas médias superiores a 4.5), as 10 piores avaliações ficam em torno de 3.5.

O Risco Sazonal e Operacional: A categoria Artigos de Natal lidera as piores avaliações, refletindo um risco clássico do e-commerce: produtos altamente sazonais não perdoam atrasos logísticos (se chegar depois do dia 25, a quebra de expectativa é total e a nota 1 é garantida). Logo na sequência, nichos como Móveis Escritório e Construção Ferramentas confirmam a dificuldade operacional de lidar com itens pesados, que têm maior taxa de quebra no transporte e exigem montagem do cliente.

O Padrão de Excelência e o Paradoxo dos Móveis: A lista verde é dominada por itens de menor complexidade logística, consumo rápido ou de caráter presenteável (Alimentos, Livros Técnicos, Papelaria, Bebidas). Contudo, a descoberta analítica mais valiosa deste painel é o contraste no departamento de mobília: enquanto Móveis Escritório desponta como uma das piores categorias da loja, Móveis Sala lidera o topo absoluto de satisfação, encostando na nota máxima (5.0). Isso sugere à plataforma que o problema não é a venda de móveis em si, mas sim a discrepância severa de qualidade ou de embalagem entre os fornecedores de linha corporativa e os de linha residencial.

In [ ]:
# --- ANÁLISE EXPLORATÓRIA: O FRETE É PROPORCIONAL AO PESO? ---

# --- PREPARAÇÃO DOS DADOS ---
# Remove linhas que não têm peso ou frete preenchidos
df_frete = df_final.dropna(subset=['product_weight_g', 'freight_value']).copy()

# --- GERAÇÃO DA VISUALIZAÇÃO COM ALTAIR ---
# Cria os pontos (cada pedido é uma bolinha)
pontos_frete = alt.Chart(df_frete).mark_circle(opacity=0.4, size=40, color='#1f77b4').encode(
    x=alt.X('product_weight_g:Q', title='Peso do Produto (gramas)'),
    y=alt.Y('freight_value:Q', title='Valor do Frete (R$)'),
    tooltip=['product_category_name:N', 'product_weight_g:Q', 'freight_value:Q']
)

# Cria a linha de regressão matemática (Tendência)
linha_tendencia = pontos_frete.transform_regression(
    'product_weight_g', 'freight_value'
).mark_line(color='red', size=3)

# Sobrepôe a linha em cima dos pontos
grafico_frete_peso = (pontos_frete + linha_tendencia).properties(
    title='Auditoria Logística: O frete é proporcional ao peso?',
    width=600,
    height=400
).interactive()

display(grafico_frete_peso)

Justificativa de Design: Para auditar se a regra de negócio básica da logística ("mais pesado = frete mais caro") é respeitada, a ferramenta visual escolhida foi o gráfico de dispersão (scatter plot) acrescido de uma linha de regressão linear. A linha vermelha nos dá a tendência matemática esperada, enquanto a dispersão dos pontos nos permite identificar rapidamente a zona de concentração das vendas e caçar anomalias no sistema de cobrança.

Ao observar o cruzamento dessas variáveis, extraí os seguintes insights:

A Zona de Conforto da Plataforma: Fica visualmente óbvio que o core business do Olist na amostra são pacotes pequenos. Existe uma nuvem densa e esmagadora no canto inferior esquerdo: produtos de até 4 kg (4.000 gramas) com fretes variando entre o valor mínimo e R$ 40.

A Regra Geral (Correlação Positiva): A linha de tendência vermelha é ascendente, confirmando matematicamente que o peso afeta o frete. Contudo, note que conforme o peso aumenta no eixo X, os pontos ficam cada vez mais espalhados e distantes da linha vermelha. Isso prova que o peso não é o único fator da equação; a distância geográfica (CEP de origem e destino) e o volume cúbico da caixa têm um peso gigantesco no preço final.

Caçando Anomalias (Os Outliers): O gráfico revela discrepâncias fascinantes que exigem atenção da equipe de operações:

Fretes Exorbitantes para Itens Leves: Existem pontos de produtos pesando menos de 8 kg, mas com fretes altíssimos, chegando a quase R$ 160 e R$ 200. Isso provavelmente reflete envios para regiões extremamente remotas do Brasil ou uso de frete expresso muito caro.

Subsídio ou Erro (O frete a R$ 0): Se você olhar a base do gráfico (eixo Y no zero), há um ponto de um produto com 14 kg (14.000 gramas) onde o frete cobrado foi exatamente R$ 0,00. Isso evidencia diretamente uma anomalia na base de dados, uma política promocional muito agressiva de "Frete Grátis" ou uma retirada em mãos.

In [ ]:
# --- ANÁLISE EXPLORATÓRIA: O RISCO DO NEGÓCIO - A CURVA DE PARETO DOS VENDEDORES ---

# --- PREPARAÇÃO DOS DADOS ---
# Soma quanto cada vendedor vendeu
df_vendedores = df_final.groupby('seller_id')['price'].sum().reset_index()

# Ordena do vendedor mais rico para o mais pobre
df_vendedores = df_vendedores.sort_values(by='price', ascending=False)

# Calcula as porcentagens acumuladas
df_vendedores['receita_acumulada'] = df_vendedores['price'].cumsum()
df_vendedores['porcentagem_receita'] = (df_vendedores['receita_acumulada'] / df_vendedores['price'].sum()) * 100

# Cria um ranking (1º, 2º, 3º vendedor...) e transforma em porcentagem do total de vendedores
df_vendedores['ranking'] = range(1, len(df_vendedores) + 1)
df_vendedores['porcentagem_vendedores'] = (df_vendedores['ranking'] / len(df_vendedores)) * 100

# --- GERAÇÃO DA VISUALIZAÇÃO COM ALTAIR ---
curva_pareto = alt.Chart(df_vendedores).mark_area(
    color='lightblue', line={'color': 'darkblue', 'size': 3}
).encode(
    x=alt.X('porcentagem_vendedores:Q', title='Porcentagem de Vendedores (%)', scale=alt.Scale(domain=[0, 100])),
    y=alt.Y('porcentagem_receita:Q', title='Porcentagem da Receita Acumulada (%)', scale=alt.Scale(domain=[0, 100])),
    tooltip=[
        alt.Tooltip('porcentagem_vendedores:Q', title='% dos Vendedores', format='.1f'),
        alt.Tooltip('porcentagem_receita:Q', title='% da Receita Gerada', format='.1f')
    ]
).properties(
    title='O Risco do Negócio: A Curva de Pareto dos Vendedores',
    width=600,
    height=400
)

# Criando linhas de referência visuais (A marca dos 20% e 80%)
linha_x = alt.Chart(pd.DataFrame({'x': [20]})).mark_rule(color='red', strokeDash=[5, 5]).encode(x='x:Q')
linha_y = alt.Chart(pd.DataFrame({'y': [80]})).mark_rule(color='red', strokeDash=[5, 5]).encode(y='y:Q')

display(curva_pareto + linha_x + linha_y)

Justificativa de Design: Para auditar a dependência da plataforma em relação aos seus maiores lojistas, utilizei um gráfico de área de distribuição acumulada. A principal escolha analítica de design foi a inclusão de linhas de referência ortogonais (em vermelho tracejado) fixadas nas coordenadas clássicas do Princípio de Pareto (20% no eixo X; 80% no eixo Y). Isso cria uma "régua visual" que permite ao leitor comparar instantaneamente a teoria econômica com a realidade do banco de dados.

Ao cruzar a curva de dados com as linhas de referência, extraí conclusões estratégicas vitais:

A Realidade vs. A Teoria (A Regra 70/20): O insight mais valioso desta visualização é que a base do Olist nesta amostra não segue a proporção exata de Pareto (80/20). Ao observarmos a linha vertical vermelha (que representa os 20% maiores vendedores), notamos que a curva azul cruza exatamente na marca de 70% da receita acumulada no eixo Y. Para que a empresa alcance a marca de 80% do faturamento, ela precisa de pouco mais de 30% da sua base de vendedores.

A Dependência do "Clube VIP": Apesar de não ser um "Pareto perfeito", a concentração de renda continua sendo altíssima. Menos de um terço dos vendedores sustenta quase todo o faturamento da plataforma. Perder apenas um desses lojistas gigantes cria um vácuo financeiro que exigiria a entrada de dezenas de vendedores comuns para ser compensado.

A Força da "Cauda Longa": Analisando a metade direita do gráfico, vemos que a curva fica quase plana. Isso representa a grande massa de pequenos lojistas (quase 70% da base) que juntos somam apenas os 20% finais do faturamento. Embora tragam pouca receita bruta individualmente, eles são essenciais para manter a diversidade do catálogo da loja (a chamada estratégia de Long Tail).

In [ ]:
# Salva o arquivo no repositório ignorando o índice numérico do pandas
df_final.to_csv(path + '/amostra_olist_consolidada.csv', index=False)